# Lab 9 GEMINI AI

In [9]:
!pip install -q google-genai langchain pypdf langchain-community

In [11]:
import os
from google import genai
from kaggle_secrets import UserSecretsClient

# Retrieve your Gemini API key securely from Kaggle Secrets
user_secrets = UserSecretsClient()
os.environ["GEMINI_API_KEY"] = user_secrets.get_secret("GEMINI_API_KEY")

# Initialize the official Google GenAI Client
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
print('Gemini client successfully initialized!')

# Make your first test API Call (Replacing Task 1.2)
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Say hello and introduce yourself!'
)

print("\n--- Test Response ---")
print(response.text)

Gemini client successfully initialized!

--- Test Response ---
Hello there!

I'm a large language model, trained by Google. It's nice to meet you!


In [12]:
from google.genai import types

# Define system behavior instructions (Replacing Task 2.2)
system_instruction = "You are a helpful assistant. Be friendly, concise, and professional. If you don't know something, say so."

# Initialize a chat session with the system persona attached
chat_session = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.7
    )
)

print('Gemini Chatbot ready! Type "quit" to exit.\n')

# Start the interactive loop (Replacing Task 2.1)
while True:
    user_input = input('You: ')
    if user_input.lower() == 'quit':
        print('Goodbye!')
        break
        
    # Send message to the session; history updates automatically
    response = chat_session.send_message(user_input)
    print(f'Assistant: {response.text}\n')

Gemini Chatbot ready! Type "quit" to exit.



You:  Artificial Intelligence


Assistant: Artificial Intelligence (AI) is a broad field of computer science dedicated to creating machines that can perform tasks that typically require human intelligence.

Key aspects include:
*   **Learning:** Acquiring information and rules for using the information.
*   **Reasoning:** Using rules to reach approximate or definite conclusions.
*   **Problem-solving:** Applying learned information to solve specific problems.
*   **Perception:** Using sensory input (like vision or speech) to understand the world.
*   **Language understanding:** Comprehending and generating human language.

Major subfields include Machine Learning (ML), Deep Learning (DL), Natural Language Processing (NLP), and Computer Vision. AI is rapidly transforming various industries and aspects of daily life.



You:  quit


Goodbye!


In [13]:
# --- HR ASSISTANT DEMO (Replacing Task 3.1) ---
hr_policy_prompt = """You are an HR assistant for a technology company.
Company policies:
Vacation: 15 days per year
Sick leave: Unlimited (with manager approval)
Remote work: 3 days per week
Health insurance: Fully covered
401(k) matching: Up to 5%

Your role:
1. Answer employee questions about policies
2. Be friendly and supportive
3. If unsure, suggest contacting HR department
4. Keep responses concise (2-3 sentences)"""

hr_chat = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(system_instruction=hr_policy_prompt, temperature=0.2)
)

print("--- Testing Gemini HR Assistant ---")
hr_questions = ['How many vacation days do I get?', 'Can I work from home?']
for q in hr_questions:
    res = hr_chat.send_message(q)
    print(f"User: {q}\nBot: {res.text}\n")


# --- CUSTOMER SUPPORT ASSISTANT DEMO (Replacing Task 3.2) ---
support_policy_prompt = """You are a customer support agent for TechShop, an electronics retailer.
Policies:
Returns: 30-day return policy
Shipping: Free over $50, otherwise $5.99
Warranty: 1 year manufacturer warranty
Support hours: 9 AM - 6 PM EST, Mon-Fri

Your tone: Empathetic and patient, apologize when appropriate, offer to escalate complex issues, and stay solution-focused."""

support_chat = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(system_instruction=support_policy_prompt, temperature=0.4)
)

print("--- Testing Gemini Customer Support Assistant ---")
support_questions = ['I want to return a product I bought 2 weeks ago', 'How much is shipping?']
for q in support_questions:
    res = support_chat.send_message(q)
    print(f"User: {q}\nBot: {res.text}\n")

--- Testing Gemini HR Assistant ---
User: How many vacation days do I get?
Bot: You get 15 vacation days per year! If you have any other questions about planning your time off, feel free to ask.

User: Can I work from home?
Bot: Yes, you can! Our company policy allows for remote work up to 3 days per week. We're happy to support a flexible work environment.

--- Testing Gemini Customer Support Assistant ---
User: I want to return a product I bought 2 weeks ago
Bot: I understand you'd like to return a product you purchased 2 weeks ago. I can certainly help you with that!

Good news – since you bought it 2 weeks ago, you are well within our 30-day return policy, so we can definitely process this for you.

To get started, could you please provide me with your order number? Once I have that, I can help you initiate the return process and provide you with a shipping label.

Typically, for a return, the item should be in its original condition and, if possible, in its original packaging. Onc

# Lab 10 RAG

In [15]:
!pip install -q google-genai langchain pypdf langchain-community langchain-text-splitters

In [21]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
# New correct import style
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create directories and mock policy text files locally
os.makedirs('company_docs', exist_ok=True)

hr_policy_content = """Employee Handbook HR Policies 
Vacation Policy: All full-time employees receive 15 days of paid vacation per year.
Vacation days accrue monthly and can be used after 90 days.
Remote Work Policy: Employees may work remotely up to 3 days per week. Remote work requires manager approval.
Parental Leave: 12 weeks paid parental leave for primary caregivers. 6 weeks paid leave for secondary caregivers."""

with open('company_docs/hr_policy.txt', 'w') as f:
    f.write(hr_policy_content)

it_policy_content = """IT and Security Policy
Dress Code: The company operates on a smart-casual dress code policy.
Device Policy: All company-issued laptops must use the corporate VPN when accessing external networks."""

with open('company_docs/it_policy.txt', 'w') as f:
    f.write(it_policy_content)

# Load data out of local files (Task 1.1)
loader = DirectoryLoader('company_docs/', glob='*.txt', loader_cls=TextLoader)
documents = loader.load()

# Slice documentation content into clear processing blocks (Task 1.2)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, separators=['\n\n', '\n', ' ', ''])
chunks = text_splitter.split_documents(documents)

print(f'Successfully structured local data into {len(chunks)} text chunks.')

Successfully structured local data into 2 text chunks.


In [20]:
def simple_search(query, chunks, top_k=3):
    query_lower = query.lower()
    scored_chunks = []
    
    for chunk in chunks:
        content_lower = chunk.page_content.lower()
        score = 0
        for word in query_lower.split():
            score += content_lower.count(word)
        if score > 0:
            scored_chunks.append((score, chunk))
            
    scored_chunks.sort(reverse=True, key=lambda x: x[0])
    return [chunk for score, chunk in scored_chunks[:top_k]]

In [22]:
def gemini_rag_query(query, chunks, top_k=2):
    # Step 1 & 2: Extract and assemble context text
    relevant_chunks = simple_search(query, chunks, top_k)
    if not relevant_chunks:
        return 'No relevant information found in documents.'
        
    context = '\n\n---\n\n'.join([chunk.page_content for chunk in relevant_chunks])
    
    # Step 3: Package context instructions for the LLM
    prompt = f"""You are a helpful assistant. Answer the question using ONLY the context provided below.
If the answer is not explicitly found in the context text, say 'I am sorry, but that information is not in the context.'

Context:
{context}

Question: {query}
Answer:"""
    
    # Step 4: Generate contextual answers from Gemini
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.0) # Zeroed out for rigid factual accuracy
    )
    return response.text

# Evaluate RAG Output Matrix
eval_questions = [
    'How many vacation days do full-time employees get?',
    'What is the dress code?',
    'What is the policy on corporate health insurance?' # Completely missing from documentation
]

print("--- Testing Gemini RAG Framework ---")
for q in eval_questions:
    print(f"\nQ: {q}")
    print(f"A: {gemini_rag_query(q, chunks)}")

--- Testing Gemini RAG Framework ---

Q: How many vacation days do full-time employees get?
A: All full-time employees receive 15 days of paid vacation per year.

Q: What is the dress code?
A: The company operates on a smart-casual dress code policy.

Q: What is the policy on corporate health insurance?
A: I am sorry, but that information is not in the context.


In [23]:
def ask_without_rag(question):
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f"You are a helpful HR assistant. Answer this: {question}"
    )
    return response.text

comp_query = 'How many vacation days do employees get?'

print('--- GEMINI WITHOUT RAG ---')
print(ask_without_rag(comp_query).strip())

print('\n--- GEMINI WITH RAG (Custom Enterprise Ground Truth) ---')
print(gemini_rag_query(comp_query, chunks).strip())

--- GEMINI WITHOUT RAG ---
That's a very important question, but the number of vacation days an employee gets can vary quite a lot! There isn't a universal standard.

To give you the most accurate answer, I'd need to know:

1.  **Which company are you asking about?** (Every company has its own policy)
2.  **What is the employee's role or level?** (Sometimes senior employees or certain roles have different accruals)
3.  **How long has the employee worked for the company?** (Many companies increase vacation days with tenure)
4.  **Is it a full-time or part-time position?** (Part-time vacation is often pro-rated)

**In general, employees typically find this information in their:**

*   **Offer Letter or Employment Contract**
*   **Company Handbook or Policy Manual**
*   **HR Portal or Benefits Website**
*   **By asking their HR department or direct manager**

If you can tell me more about the specific situation, I might be able to guide you further!

--- GEMINI WITH RAG (Custom Enterprise